In [ ]:
# K-00 Run configuration
# Edit this cell before each Kaggle run.
# For a smoke-real-run, use DATASETS_TO_RUN = ["pathmnist"], PHASE1_EPOCHS = 1, PHASE2_EPOCHS = 1.
# For a valid Kaggle submission, use DATASETS_TO_RUN = "all".

RUN_NAME = "smoke_pathmnist_resnet18"
DATASETS_TO_RUN = ["pathmnist"]  # "all" or list like ["pathmnist", "dermamnist"]

MODEL_NAME = "resnet18"          # "resnet18" | "efficientnetv2_s" | etc.
IMAGE_SIZE = 224
PHASE1_EPOCHS = 1
PHASE2_EPOCHS = 1
PATIENCE = 7

BACKBONE_LR = 1e-4
HEAD_LR = 1e-3
WEIGHT_DECAY = 1e-4
BATCH_SIZE = 32
AUG_LEVEL = "medium"             # "light" | "medium" | "heavy"

LOSS_TYPE = "cross_entropy"      # "cross_entropy" | "weighted_ce" | "focal"
FOCAL_GAMMA = 2.0

SCHEDULER = "cosine"             # "cosine" only; OneCycleLR needs per-batch stepping
USE_TRAINVAL = False             # True only for final small-dataset runs after tuning
TTA_ENABLED = True
SEED = 42

print(f"Run: {RUN_NAME}")
print(f"Datasets: {DATASETS_TO_RUN}")
print(f"Model: {MODEL_NAME} | Image size: {IMAGE_SIZE}")
print(f"Loss: {LOSS_TYPE} | Aug: {AUG_LEVEL}")
print(f"Phase1: {PHASE1_EPOCHS} | Phase2: {PHASE2_EPOCHS} | Patience: {PATIENCE}")
print(f"LR backbone: {BACKBONE_LR} | LR head: {HEAD_LR}")


In [ ]:
# K-01 Install dependencies and verify runtime
# Run once per Kaggle session.

import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "timm",
    "scikit-learn",
    "scipy",
    "pandas",
    "tqdm",
])

import torch
import timm

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

model_check = timm.create_model("resnet18", pretrained=False, num_classes=2, in_chans=3)
print("timm OK — model creation works")
del model_check


In [ ]:
# K-02 Clone repository and import project modules
# The repo must be public OR Kaggle must have authenticated GitHub access.

import os
import sys
import subprocess
from pathlib import Path

REPO_NAME = "ACCV"
REPO_URL = "https://github.com/SamiiraEnache/ACCV.git"

WORKING_DIR = Path("/kaggle/working")
REPO_DIR = WORKING_DIR / REPO_NAME
WORKING_DIR.mkdir(parents=True, exist_ok=True)

if not REPO_DIR.exists():
    print(f"Cloning {REPO_URL} into {REPO_DIR} ...")
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])
else:
    print(f"Repository already exists at {REPO_DIR}; pulling latest changes...")
    subprocess.check_call(["git", "-C", str(REPO_DIR), "pull"])

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
os.chdir(REPO_DIR)

from configs.dataset_configs import DATASET_CONFIGS, FOCUS_TUNING_DATASETS, NO_VFLIP_DATASETS, SMALL_DATASETS
import configs.base_config as bc
from src.dataset import get_dataloaders
from src.transforms import get_train_transform, get_val_transform
from src.models import build_model, freeze_backbone, unfreeze_all, get_optimizer_phase2
from src.train import train_one_epoch, validate, compute_class_weights
from src.evaluate import predict_dataset, compute_harmonic_mean
from src.submission import build_submission_csv

print("All imports OK")
print(f"Repo cwd: {Path.cwd()}")


In [ ]:
# K-03 Reproducibility and output directories
# Also patch the Kaggle competition dataset path discovered during real execution.

import os
import random
from pathlib import Path
from datetime import datetime, UTC

import numpy as np
import pandas as pd
import torch
from scipy.stats import hmean
from sklearn.metrics import f1_score
from torch import nn
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.auto import tqdm

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
bc.BASE_CONFIG["device"] = DEVICE

CKPT_DIR = Path(f"/kaggle/working/checkpoints/{RUN_NAME}")
RESULTS_DIR = Path("/kaggle/working/results")
SUBMISSION_DIR = Path("/kaggle/working/submissions")

for directory in [CKPT_DIR, RESULTS_DIR, SUBMISSION_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Device: {DEVICE} | Seed: {SEED}")
print(f"Data root: {bc.BASE_CONFIG['data_root']}")
print(f"Checkpoint dir: {CKPT_DIR}")
print(f"Results dir: {RESULTS_DIR}")
print(f"Submission dir: {SUBMISSION_DIR}")

results_rows = []


In [ ]:
# K-04 Verify data access
# Confirms the selected datasets are present and readable.

required_keys = {"train_images", "train_labels", "val_images", "val_labels", "test_images"}

if DATASETS_TO_RUN == "all":
    datasets_list = list(DATASET_CONFIGS.keys())
else:
    datasets_list = list(DATASETS_TO_RUN)

print(f"Datasets selected: {datasets_list}")

for dataset_name in datasets_list:
    npz_path = Path(bc.BASE_CONFIG["data_root"]) / f"{dataset_name}.npz"
    assert npz_path.exists(), f"Missing dataset file: {npz_path}"

    with np.load(npz_path) as data:
        missing_keys = required_keys - set(data.files)
        assert not missing_keys, f"{dataset_name} missing keys: {missing_keys}"

        print(
            f"{dataset_name}: "
            f"train={data['train_images'].shape[0]:,} "
            f"val={data['val_images'].shape[0]:,} "
            f"test={data['test_images'].shape[0]:,} "
            f"shape={data['train_images'].shape[1:]}"
        )

print("Selected datasets present and readable.")


In [ ]:
# K-05 Training loop
# Trains all selected datasets independently and saves best + last checkpoints/probability arrays.

import torch.nn.functional as F

class FocalLoss(nn.Module):
    def __init__(self, gamma: float = 2.0, weight=None):
        super().__init__()
        self.gamma = gamma
        self.weight = weight

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weight, reduction="none")
        pt = torch.exp(-ce)
        return (((1.0 - pt) ** self.gamma) * ce).mean()


def make_criterion(dataset_name: str, n_classes: int):
    labels = np.load(Path(bc.BASE_CONFIG["data_root"]) / f"{dataset_name}.npz")["train_labels"]

    if LOSS_TYPE == "weighted_ce":
        weights = compute_class_weights(labels, n_classes).to(DEVICE)
        print(f"Loss: WeightedCE | weights={weights.detach().cpu().numpy().round(2)}")
        return nn.CrossEntropyLoss(weight=weights)

    if LOSS_TYPE == "focal":
        weights = compute_class_weights(labels, n_classes).to(DEVICE)
        print(f"Loss: FocalLoss(gamma={FOCAL_GAMMA}) with class weights")
        return FocalLoss(gamma=FOCAL_GAMMA, weight=weights)

    print("Loss: CrossEntropyLoss")
    return nn.CrossEntropyLoss()


all_predictions = {}
all_val_f1 = {}
training_outputs = {}

print(f"Training {len(datasets_list)} datasets: {datasets_list}")
print("=" * 80)

for ds_name in datasets_list:
    cfg = DATASET_CONFIGS[ds_name]
    print("\n" + "=" * 80)
    print(
        f"DATASET: {ds_name.upper()} | classes={cfg['n_classes']} | "
        f"imbalance={cfg['imbalance_ratio']:.1f}x"
    )
    print("=" * 80)

    train_tf = get_train_transform(IMAGE_SIZE, AUG_LEVEL, cfg["is_rgb"], ds_name)
    val_tf = get_val_transform(IMAGE_SIZE, cfg["is_rgb"])

    use_tv = USE_TRAINVAL and ds_name in SMALL_DATASETS
    train_loader, val_loader, test_loader = get_dataloaders(
        ds_name,
        IMAGE_SIZE,
        BATCH_SIZE,
        train_tf,
        val_tf,
        num_workers=2,
        use_trainval=use_tv,
    )

    print(
        f"Loaders ready | train={len(train_loader.dataset):,} "
        f"val={len(val_loader.dataset):,} test={len(test_loader.dataset):,}"
        + (" [train+val merged]" if use_tv else "")
    )

    criterion = make_criterion(ds_name, cfg["n_classes"])
    model = build_model(MODEL_NAME, cfg["n_classes"]).to(DEVICE)

    if PHASE1_EPOCHS > 0:
        freeze_backbone(model)
        opt_p1 = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=HEAD_LR,
            weight_decay=WEIGHT_DECAY,
        )

        print(f"\nPhase 1: {PHASE1_EPOCHS} epochs (frozen backbone)")
        for epoch in range(PHASE1_EPOCHS):
            tl, tf1 = train_one_epoch(model, train_loader, opt_p1, criterion, DEVICE)
            vl, vf1, _ = validate(model, val_loader, criterion, DEVICE)
            print(
                f"  [{ds_name}] P1 Ep {epoch + 1}/{PHASE1_EPOCHS} "
                f"| train_loss={tl:.4f} | train_f1={tf1:.4f} | val_f1={vf1:.4f}"
            )
    else:
        print("\nPhase 1 skipped (PHASE1_EPOCHS=0)")

    unfreeze_all(model)
    opt_p2 = get_optimizer_phase2(model, BACKBONE_LR, HEAD_LR, WEIGHT_DECAY)

    if SCHEDULER != "cosine":
        raise ValueError("Only cosine scheduler is enabled in this template.")
    scheduler = CosineAnnealingLR(opt_p2, T_max=PHASE2_EPOCHS, eta_min=1e-6)

    best_val_f1 = 0.0
    best_epoch = 0
    epochs_no_improve = 0
    MIN_EPOCHS = min(10, PHASE2_EPOCHS)

    best_ckpt_path = CKPT_DIR / f"{ds_name}_best_val_f1.pth"
    last_ckpt_path = CKPT_DIR / f"{ds_name}_last_epoch.pth"

    print(f"\nPhase 2: {PHASE2_EPOCHS} epochs (full fine-tuning)")

    for epoch in range(PHASE2_EPOCHS):
        tl, tf1 = train_one_epoch(model, train_loader, opt_p2, criterion, DEVICE)
        vl, vf1, pc_f1 = validate(model, val_loader, criterion, DEVICE)

        scheduler.step()

        improved = vf1 > best_val_f1
        marker = " ← BEST" if improved else ""
        print(
            f"  [{ds_name}] P2 Ep {epoch + 1}/{PHASE2_EPOCHS} "
            f"| train_loss={tl:.4f} | train_f1={tf1:.4f} | "
            f"val_loss={vl:.4f} | val_f1={vf1:.4f}{marker}"
        )

        if improved:
            best_val_f1 = vf1
            best_epoch = epoch + 1
            epochs_no_improve = 0
            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "epoch": epoch + 1,
                    "val_f1": vf1,
                    "per_class_f1": pc_f1.tolist(),
                    "config": {
                        "run_name": RUN_NAME,
                        "dataset": ds_name,
                        "model": MODEL_NAME,
                        "image_size": IMAGE_SIZE,
                        "backbone_lr": BACKBONE_LR,
                        "head_lr": HEAD_LR,
                        "weight_decay": WEIGHT_DECAY,
                        "loss": LOSS_TYPE,
                        "aug_level": AUG_LEVEL,
                        "scheduler": SCHEDULER,
                    },
                },
                best_ckpt_path,
            )
        else:
            epochs_no_improve += 1

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "epoch": epoch + 1,
                "val_f1": vf1,
            },
            last_ckpt_path,
        )

        if epochs_no_improve >= PATIENCE and (epoch + 1) >= MIN_EPOCHS:
            print(f"  Early stopping at epoch {epoch + 1}")
            break

    best_ckpt = torch.load(best_ckpt_path, map_location=DEVICE)
    model.load_state_dict(best_ckpt["model_state_dict"])
    model.eval()

    test_preds, test_probs_best = predict_dataset(model, test_loader, DEVICE)
    _, val_probs_best = predict_dataset(model, val_loader, DEVICE)

    np.save(CKPT_DIR / f"{ds_name}_test_probs_best.npy", test_probs_best)
    np.save(CKPT_DIR / f"{ds_name}_val_probs_best.npy", val_probs_best)

    last_ckpt = torch.load(last_ckpt_path, map_location=DEVICE)
    model.load_state_dict(last_ckpt["model_state_dict"])
    model.eval()
    _, test_probs_last = predict_dataset(model, test_loader, DEVICE)
    _, val_probs_last = predict_dataset(model, val_loader, DEVICE)

    np.save(CKPT_DIR / f"{ds_name}_test_probs_last.npy", test_probs_last)
    np.save(CKPT_DIR / f"{ds_name}_val_probs_last.npy", val_probs_last)

    model.load_state_dict(best_ckpt["model_state_dict"])
    model.eval()

    all_predictions[ds_name] = test_preds
    all_val_f1[ds_name] = best_val_f1

    training_outputs[ds_name] = {
        "checkpoint_path": str(best_ckpt_path),
        "last_checkpoint_path": str(last_ckpt_path),
        "test_loader": test_loader,
        "val_loader": val_loader,
        "best_val_f1": best_val_f1,
        "best_epoch": best_epoch,
    }

    test_f1_sanity = None
    npz_path = Path(bc.BASE_CONFIG["data_root"]) / f"{ds_name}.npz"
    with np.load(npz_path) as data:
        if "test_labels" in data.files:
            test_labels = data["test_labels"].squeeze()
            test_f1_sanity = f1_score(test_labels, test_preds, average="macro", zero_division=0)

    print(
        f"\n  [{ds_name}] FINAL | best_val_f1={best_val_f1:.4f} "
        f"| test_f1_sanity={test_f1_sanity if test_f1_sanity is not None else 'NA'}"
    )

    results_rows.append(
        {
            "dataset": ds_name,
            "model_name": MODEL_NAME,
            "image_size": IMAGE_SIZE,
            "lr_backbone": BACKBONE_LR,
            "lr_head": HEAD_LR,
            "weight_decay": WEIGHT_DECAY,
            "scheduler": SCHEDULER,
            "aug_level": AUG_LEVEL,
            "loss_type": LOSS_TYPE,
            "epochs": best_epoch,
            "val_macro_f1": round(float(best_val_f1), 4),
            "notes": RUN_NAME,
        }
    )

print("\n" + "=" * 80)
print("SELECTED DATASETS TRAINED")
print("=" * 80)


In [ ]:
# K-06 Harmonic mean summary
# For partial runs, this is the harmonic mean over the selected datasets only.

print("\nPer-dataset validation F1:")
print("-" * 50)

for ds, score in all_val_f1.items():
    flag = " ← WEAK" if score < 0.70 else ""
    print(f"  {ds:<18} {score:.4f}{flag}")

if len(all_val_f1) > 0 and all(score > 0 for score in all_val_f1.values()):
    harmonic_mean_f1 = float(hmean(list(all_val_f1.values())))
else:
    harmonic_mean_f1 = 0.0

arithmetic_mean_f1 = float(np.mean(list(all_val_f1.values()))) if all_val_f1 else 0.0

print("-" * 50)
print(f"  HARMONIC MEAN:    {harmonic_mean_f1:.4f}")
print(f"  ARITHMETIC MEAN:  {arithmetic_mean_f1:.4f}")
print("\nNote: final competition metric requires all 11 datasets.")


In [ ]:
# K-07 Submission CSV
# TTA averages softmax probabilities across deterministic image variants.
# For partial smoke runs, final Kaggle submission CSV is skipped because competition submissions require all 11 datasets.

def load_best_model(dataset_name):
    model = build_model(MODEL_NAME, DATASET_CONFIGS[dataset_name]["n_classes"])
    checkpoint = torch.load(training_outputs[dataset_name]["checkpoint_path"], map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(DEVICE)
    model.eval()
    return model


def predict_with_tta(model, loader, dataset_name):
    all_probabilities = []
    model.eval()

    with torch.no_grad():
        for batch in tqdm(loader, desc=f"Predict {dataset_name}"):
            images = batch[0] if isinstance(batch, (list, tuple)) else batch
            images = images.to(DEVICE)

            logits_list = [model(images)]

            if TTA_ENABLED:
                logits_list.append(model(torch.flip(images, dims=[3])))

                if dataset_name not in NO_VFLIP_DATASETS:
                    logits_list.append(model(torch.flip(images, dims=[2])))

            probabilities = torch.stack(
                [torch.softmax(logits, dim=1) for logits in logits_list],
                dim=0,
            ).mean(dim=0)

            all_probabilities.append(probabilities.cpu().numpy())

    probabilities = np.concatenate(all_probabilities, axis=0)
    return probabilities.argmax(axis=1), probabilities


test_predictions = {}
test_probabilities = {}

for dataset_name in datasets_list:
    model = load_best_model(dataset_name)
    predictions, probabilities = predict_with_tta(
        model,
        training_outputs[dataset_name]["test_loader"],
        dataset_name,
    )
    test_predictions[dataset_name] = predictions
    test_probabilities[dataset_name] = probabilities
    print(dataset_name, predictions.shape, probabilities.shape)


timestamp = datetime.now(UTC).strftime("%Y%m%d_%H%M%S")
expected_datasets = set(DATASET_CONFIGS.keys())
provided_datasets = set(test_predictions.keys())

if provided_datasets != expected_datasets:
    missing = sorted(expected_datasets - provided_datasets)
    extra = sorted(provided_datasets - expected_datasets)

    print("=" * 80)
    print("PARTIAL RUN DETECTED — skipping final Kaggle submission CSV.")
    print("This is expected for smoke runs such as DATASETS_TO_RUN = ['pathmnist'].")
    print(f"Provided datasets: {sorted(provided_datasets)}")
    print(f"Missing datasets: {missing}")
    print(f"Extra datasets: {extra}")
    print("To create a valid competition submission, rerun with DATASETS_TO_RUN = 'all'.")
    print("=" * 80)

    submission_path = None
    submission_df = None

else:
    submission_path = SUBMISSION_DIR / f"submission_{RUN_NAME}_{timestamp}.csv"
    submission_df = build_submission_csv(test_predictions, submission_path)

    assert list(submission_df.columns) == ["id", "id_image_in_task", "task_name", "label"]
    assert submission_df["id"].is_monotonic_increasing
    assert set(submission_df["task_name"].unique()) == expected_datasets

    print("Saved submission:", submission_path)
    display(submission_df.head())


In [ ]:
# K-08 Save results/config
# Works for both full runs and partial smoke runs.

import json
import shutil

def safe_get(name, default=None):
    return globals().get(name, default)


export_dir = WORKING_DIR / f"exports_{RUN_NAME}_{timestamp}"
export_dir.mkdir(parents=True, exist_ok=True)

results_path = RESULTS_DIR / f"results_{RUN_NAME}_{timestamp}.csv"
results_df = pd.DataFrame(results_rows)
results_df.to_csv(results_path, index=False)

run_config = {
    "run_name": safe_get("RUN_NAME"),
    "timestamp": timestamp,
    "datasets_to_run": safe_get("DATASETS_TO_RUN"),
    "datasets_resolved": datasets_list,
    "datasets_completed": list(test_predictions.keys()),
    "model_name": safe_get("MODEL_NAME"),
    "image_size": safe_get("IMAGE_SIZE"),
    "phase1_epochs": safe_get("PHASE1_EPOCHS"),
    "phase2_epochs": safe_get("PHASE2_EPOCHS"),
    "patience": safe_get("PATIENCE"),
    "backbone_lr": safe_get("BACKBONE_LR"),
    "head_lr": safe_get("HEAD_LR"),
    "weight_decay": safe_get("WEIGHT_DECAY"),
    "batch_size": safe_get("BATCH_SIZE"),
    "aug_level": safe_get("AUG_LEVEL"),
    "loss_type": safe_get("LOSS_TYPE"),
    "focal_gamma": safe_get("FOCAL_GAMMA") if safe_get("LOSS_TYPE") == "focal" else None,
    "scheduler": safe_get("SCHEDULER"),
    "seed": safe_get("SEED"),
    "use_trainval": safe_get("USE_TRAINVAL"),
    "tta_enabled": safe_get("TTA_ENABLED"),
    "harmonic_mean_val": float(harmonic_mean_f1),
    "partial_run": submission_path is None,
    "data_root": bc.BASE_CONFIG["data_root"],
    "repo_url": REPO_URL,
}

config_path = RESULTS_DIR / f"run_config_{RUN_NAME}_{timestamp}.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(run_config, f, indent=2)

submission_name = (
    submission_path.name
    if submission_path is not None
    else "NO_SUBMISSION_PARTIAL_RUN"
)

submission_log = pd.DataFrame(
    [
        {
            "version": RUN_NAME,
            "timestamp": timestamp,
            "public_score": "",
            "key_changes": f"{MODEL_NAME}, {LOSS_TYPE}, TTA={TTA_ENABLED}",
            "notes": (
                f"validation_harmonic_mean={harmonic_mean_f1:.6f}; "
                f"submission={submission_name}"
            ),
        }
    ]
)

submission_log_path = RESULTS_DIR / f"submission_log_{RUN_NAME}_{timestamp}.csv"
submission_log.to_csv(submission_log_path, index=False)

files_to_copy = [
    results_path,
    config_path,
    submission_log_path,
]

if submission_path is not None:
    files_to_copy.append(submission_path)

for path in files_to_copy:
    shutil.copy2(path, export_dir / Path(path).name)

print("=" * 80)
print("EXPORT COMPLETE")
print("=" * 80)
print(f"Export directory: {export_dir}")
print()
print("Files saved:")
for path in sorted(export_dir.iterdir()):
    print(" -", path.name)

print()
print("=" * 80)
print("FILES TO DOWNLOAD TO LOCAL")
print("=" * 80)
print(f"[REQUIRED] {results_path}")
print(f"[REQUIRED] {config_path}")
print(f"[REQUIRED] {submission_log_path}")

if submission_path is not None:
    print(f"[REQUIRED] {submission_path}")
else:
    print("[SKIPPED] No submission CSV created because this was a partial smoke run.")

print()
print("=" * 80)
print("FILES TO NOT COMMIT")
print("=" * 80)
print("*.pth checkpoint files")
print("raw data")
print("large .npy probability files unless explicitly needed for TASK-009")


In [ ]:
# K-09 Manual Kaggle submit
# Run only for full 11-dataset runs, after K-07 created a valid submission CSV.

if submission_path is None:
    print("No submission_path exists because this was a partial run. Skipping Kaggle submit.")
else:
    print(f"Ready to submit: {submission_path}")
    print("Uncomment the subprocess.run block below when you want to submit.")

    # import subprocess
    # result = subprocess.run(
    #     [
    #         "kaggle",
    #         "competitions",
    #         "submit",
    #         "-c",
    #         "tensor-reloaded-multi-task-med-mnist",
    #         "-f",
    #         str(submission_path),
    #         "-m",
    #         f"{RUN_NAME} hm_val={harmonic_mean_f1:.4f}",
    #     ],
    #     capture_output=True,
    #     text=True,
    # )
    # print(result.stdout)
    # if result.returncode != 0:
    #     print("ERROR:", result.stderr)
